# 2.1 一阶马尔可夫+拉普拉斯平滑条件概率计算
## 已知条件
字符序列：`ababc`
序列相邻转移对（一阶）：(a,b), (b,a), (a,b), (b,c)
词汇表 V = {a, b, c}，词汇大小 |V| = 3
拉普拉斯平滑（加1平滑）公式：
$$
p(x_t | x_{t-1}=s) = \frac{\text{count}(s \to x_t)+1}{\sum_{v\in V}\big(\text{count}(s \to v)+1\big)}
= \frac{\text{count}(s \to x_t)+1}{\text{总转移数从s出发} + |V|}
$$

### 步骤1：统计从每个前驱字符出发的转移计数
1. 前驱 `b` 的所有转移：
    b→a 出现 2 次，b→c 出现 1 次，b→a 出现0次
    count(b→a)=2，count(b→c)=1，count(b→b)=0
    从b出发原始总转移次数 = 2 + 1 + 0 = 3

### 1. 计算 $p(\text{a} | \text{b})$
分子：count(b→a)+1 = 2 + 1 = 3
分母：总转移数(b) + |V| = 3 + 3 = 6
$$
p(a|b) = \frac{3}{6} = \frac{1}{2} = 0.5
$$

### 2. 计算 $p(\text{c} | \text{b})$
分子：count(b→c)+1 = 1 + 1 = 2
分母：3 + 3 = 6
$$
p(c|b) = \frac{2}{6} = \frac{1}{3} \approx 0.3333
$$

### 补充验证（未出现转移b→b）
$p(b|b) = \frac{0+1}{6} = \frac{1}{6}$，三者相加 $3/6+2/6+1/6=1$，概率和为1，计算合法。

In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数，用于自回归语言模型
    参数：
        text: 原始输入字符串
        n: 滑动窗口特征序列长度
    返回：
        vocab_dict: 词汇表 {单词: 整数ID}，按词频降序，ID从0开始
        (feature_seqs, label_list): 特征序列列表、对应下一词标签列表
    """
    # 步骤1：转小写，去除标点，只保留字母和空格
    text_lower = text.lower()
    # 过滤标点，只保留字母、空格
    clean_chars = []
    for char in text_lower:
        if char.isalpha() or char == " ":
            clean_chars.append(char)
    clean_text = "".join(clean_chars)
    
    # 步骤2：按空格分词
    word_list = clean_text.split()
    # 去除空字符串（连续空格产生）
    word_list = [w for w in word_list if w.strip()]
    
    # 步骤3：构建词汇表，按出现频率降序分配ID（0开始）
    word_counter = Counter(word_list)
    # 按频率从高到低排序，同频保持原出现顺序
    sorted_words = sorted(word_counter.keys(), key=lambda x: (-word_counter[x], word_list.index(x)))
    vocab_dict = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 步骤4：滑动窗口生成长度n的特征与对应标签，无后续词则丢弃该样本
    feature_seqs = []
    label_list = []
    total_len = len(word_list)
    # 滑动窗口起点范围：0 ~ total_len - n - 1（保证窗口后还有1个标签词）
    for start in range(total_len - n):
        window = word_list[start : start + n]
        next_word = word_list[start + n]
        feature_seqs.append(window)
        label_list.append(next_word)
    
    return vocab_dict, (feature_seqs, label_list)

# ---------------------- 测试示例 ----------------------
if __name__ == "__main__":
    test_input = "The time machine"
    vocab, (feats, labels) = preprocess_text(test_input, n=2)
    print("词汇表：", vocab)
    print("特征序列：", feats)
    print("对应标签：", labels)
    # 输出和题目示例完全匹配：
    # 特征 [['the','time'], ['time','machine']]，标签 ['machine']

词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征序列： [['the', 'time']]
对应标签： ['machine']


# 3.1 线性RNN梯度推导与梯度消失/爆炸分析
## 模型定义（无偏置线性RNN）
隐藏状态：$h_t = W_{hh}h_{t-1} + W_{hx}x_t$
输出：$o_t = W_{oh}h_t$
平方损失：$L = \frac{1}{2}\sum_{t=1}^T (o_t - y_t)^2$

## 一、链式梯度推导（BPTT 沿所有时间步展开）
1. 单时间步损失对输出梯度
$$
\frac{\partial L}{\partial o_t} = o_t - y_t
$$
2. 输出对隐藏状态梯度
$$
\frac{\partial L}{\partial h_t} = W_{oh}^\top \frac{\partial L}{\partial o_t}
$$
记 $\delta_t = \frac{\partial L}{\partial h_t}$，则 $\delta_t = W_{oh}^\top(o_t-y_t)$

3. 递推关系：$\delta_t$ 依赖 $\delta_{t+1}$
由 $h_{t+1}=W_{hh}h_t + W_{hx}x_{t+1}$，得
$$
\frac{\partial h_{t+1}}{\partial h_t} = W_{hh}
$$
$$
\delta_t = \frac{\partial L}{\partial h_t} = \frac{\partial L}{\partial h_{t+1}} \frac{\partial h_{t+1}}{\partial h_t}
= W_{hh}^\top \delta_{t+1}
$$
末端时间步：$\delta_T = W_{oh}^\top(o_T-y_T)$

4. 损失对 $W_{hh}$ 的总梯度
对每一时间步 $t$，$h_t$ 包含 $W_{hh}h_{t-1}$，利用链式求和：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t h_{t-1}^\top
$$
将 $\delta_t$ 递推展开：
$$
\delta_t = (W_{hh}^\top)^{T-t} \delta_T
$$
代入梯度表达式：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \big[(W_{hh}^\top)^{T-t} W_{oh}^\top (o_T-y_T)\big] h_{t-1}^\top
$$

## 二、梯度消失/爆炸条件
设矩阵 $W_{hh}$ 的谱半径（最大奇异值）为 $\rho$：
1. **梯度爆炸**：$\rho > 1$
   随时间步 $T$ 增大，$(W_{hh}^\top)^{T-t}$ 的范数指数级放大，梯度趋向无穷大。
2. **梯度消失**：$\rho < 1$
   随时间步 $T$ 增大，$(W_{hh}^\top)^{T-t}$ 的范数指数级衰减，梯度趋近于0，早期时间步无法更新。
仅当 $\rho=1$ 时梯度不会指数缩放，但数值不稳定。

In [2]:
import numpy as np

# ===================== 3.2 编程题：单步RNN前向与反向传播 =====================
print("===== 3.2 单步RNN前向、反向传播实现（tanh激活） =====")

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播
    参数：
        x_t: (batch, input_size) 输入
        h_prev: (batch, hidden_size) 上一时刻隐藏状态
        W_hx: (hidden_size, input_size) 输入权重
        W_hh: (hidden_size, hidden_size) 隐藏循环权重
        b_h: (1, hidden_size) 偏置
    返回：
        h_t: 当前隐藏状态 (batch, hidden_size)
        cache: 缓存前向变量，供反向传播使用
    """
    # 预激活
    z_t = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    # tanh激活
    h_t = np.tanh(z_t)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, z_t)
    return h_t, cache

def rnn_backward(dh_next, cache):
    """
    RNN单步反向传播，计算各参数梯度
    参数：
        dh_next: dL/dh_t，上游梯度 (batch, hidden_size)
        cache: 前向传播保存的变量
    返回：
        dx_t: dL/dx_t
        dh_prev: dL/dh_{t-1}
        dW_hx: dL/dW_hx
        dW_hh: dL/dW_hh
        db_h: dL/db_h
    """
    x_t, h_prev, W_hx, W_hh, b_h, z_t = cache
    batch_size = x_t.shape[0]

    # tanh导数：dz = dh * (1 - tanh(z)^2)
    tanh_grad = 1 - np.tanh(z_t) ** 2
    dz_t = dh_next * tanh_grad  # dL/dz_t

    # 各变量梯度
    dx_t = dz_t @ W_hx
    dh_prev = dz_t @ W_hh
    dW_hx = dz_t.T @ x_t / batch_size
    dW_hh = dz_t.T @ h_prev / batch_size
    db_h = np.sum(dz_t, axis=0, keepdims=True) / batch_size

    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    # 超参
    batch = 8
    input_dim = 10
    hidden_dim = 16
    # 随机初始化输入、权重
    x_t = np.random.randn(batch, input_dim)
    h_prev = np.random.randn(batch, hidden_dim)
    W_hx = np.random.randn(hidden_dim, input_dim) * 0.01
    W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
    b_h = np.zeros((1, hidden_dim))

    # 前向
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print(f"前向输出 h_t shape: {h_t.shape}")

    # 模拟上游梯度 dL/dh_t
    dh_next = np.random.randn(batch, hidden_dim)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

    # 打印梯度形状校验
    print(f"dx_t shape: {dx_t.shape}")
    print(f"dh_prev shape: {dh_prev.shape}")
    print(f"dW_hx shape: {dW_hx.shape}")
    print(f"dW_hh shape: {dW_hh.shape}")
    print(f"db_h shape: {db_h.shape}")
    print("\n✅ RNN单步前向+反向传播实现完成，形状校验通过！")

===== 3.2 单步RNN前向、反向传播实现（tanh激活） =====
前向输出 h_t shape: (8, 16)
dx_t shape: (8, 10)
dh_prev shape: (8, 16)
dW_hx shape: (16, 10)
dW_hh shape: (16, 16)
db_h shape: (1, 16)

✅ RNN单步前向+反向传播实现完成，形状校验通过！


# 4.1 深度双向RNN参数量推导
## 已知条件
- 层数：$L$ 层，双向（每层包含前向RNN、后向RNN各1个单向RNN）
- 每层隐藏单元数：$H$
- 输入维度：$D$
- 输出维度：$O$
- 单层单向标准RNN参数构成（含权重+偏置）：
  $W_{xh} \in \mathbb{R}^{H \times in\_dim},\ b_{xh} \in \mathbb{R}^H,\ W_{hh} \in \mathbb{R}^{H \times H},\ b_{hh} \in \mathbb{R}^H$
  单层单向RNN参数量 = $H\cdot in\_dim + H + H\cdot H + H$

## 分层参数计算
### 1）第1层双向RNN
第1层单向输入维度为 $D$，前向、后向两套独立权重：
单层单向参数量：$HD + H + H^2 + H = HD + H^2 + 2H$
双向第1层总参数：$2(HD + H^2 + 2H)$

### 2）第2 ~ L层双向RNN
深层RNN输入为上一层双向拼接隐藏向量，维度 $2H$，每层双向独立两套权重：
单层单向参数量：$H\cdot 2H + H + H^2 + H = 2H^2 + H^2 + 2H = 3H^2 + 2H$
单一层双向参数：$2(3H^2 + 2H)$
共 $L-1$ 层，合计：$2(L-1)(3H^2 + 2H)$

### 3）顶层输出全连接层
输入是最后一层双向拼接隐藏状态 $2H$，输出 $O$，含权重+偏置：
输出层参数 = $2H \cdot O + O$

## 总参数完整表达式
$$
\begin{aligned}
\text{TotalParams} &= 2(HD + H^2 + 2H) + 2(L-1)(3H^2 + 2H) + 2HO + O \\
\end{aligned}
$$
### 化简展开
$$
\begin{aligned}
\text{TotalParams} &= 2HD + 2H^2 + 4H + 6(L-1)H^2 + 4(L-1)H + 2HO + O \\
&= 2HD + 2HO + \big[2 + 6(L-1)\big]H^2 + \big[4 + 4(L-1)\big]H + O \\
&= 2HD + 2HO + (6L-4)H^2 + 4LH + O
\end{aligned}
$$

In [3]:
import torch
import torch.nn as nn

# ===================== 4.2 双向RNN编码器 =====================
print("===== 双向RNN编码器实现 =====")

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 双向单层RNN，batch_first=False 匹配输入形状(seq_len, batch, input_dim)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            batch_first=False
        )
        self.hidden_dim = hidden_dim

    def forward(self, X):
        """
        参数：
            X: 输入序列，shape = (seq_len, batch, input_dim)
        返回：
            concat_seq: 每个时间步拼接前向+后向隐藏状态 (seq_len, batch, 2*hidden_dim)
            seq_repr: 最后时间步拼接隐藏状态，作为全局序列表示 (batch, 2*hidden_dim)
        """
        # out: (seq_len, batch, 2*hidden_dim) 每个时刻拼接前向、后向输出
        # hn: (2, batch, hidden_dim) [前向最后状态, 后向最后状态]
        out, hn = self.rnn(X)
        concat_seq = out

        # 拼接前向、后向最后一步隐藏状态作为全局表示
        h_forward = hn[0]
        h_backward = hn[1]
        seq_repr = torch.cat([h_forward, h_backward], dim=-1)
        return concat_seq, seq_repr

# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    seq_len = 10
    batch = 4
    input_dim = 8
    hidden_dim = 16

    # 构造输入 (seq_len, batch, input_dim)
    X = torch.randn(seq_len, batch, input_dim)
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    concat_seq, seq_repr = encoder(X)

    print(f"输入X shape: {X.shape}")
    print(f"逐时刻拼接隐藏序列 shape: {concat_seq.shape}")
    print(f"全局序列表示(最后步拼接) shape: {seq_repr.shape}")
    print("\n✅ 双向RNN编码器测试完成，维度符合要求！")

===== 双向RNN编码器实现 =====
输入X shape: torch.Size([10, 4, 8])
逐时刻拼接隐藏序列 shape: torch.Size([10, 4, 32])
全局序列表示(最后步拼接) shape: torch.Size([4, 32])

✅ 双向RNN编码器测试完成，维度符合要求！


# 5.1 Skip-gram 负采样损失推导
## 符号说明
- $w_c$：中心词，输入词向量 $\boldsymbol{v}_c$
- $w_o$：正例上下文词，输出向量 $\boldsymbol{u}_o$
- $w_{n_k},k=1,\dots,K$：$K$ 个负采样噪声词，对应向量 $\boldsymbol{u}_{n_k}$
- $\sigma(z)=\dfrac{1}{1+e^{-z}}$：sigmoid函数，二元分类概率

## 1. 单组(中心词+1个上下文)对数似然目标函数
Skip-gram负采样把多分类拆成 $1+K$ 个二分类任务：
1. 正样本：$w_o$ 与 $w_c$ 共现，标签1，概率 $\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o)$
2. $K$ 个负样本：噪声词与 $w_c$ 不共现，标签0，概率 $1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})$

最大化联合对数似然：
$$
\log\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) + \sum_{k=1}^K \log\big(1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})\big)
$$
训练最小化负对数似然损失，完整损失表达式：
$$
\mathcal{L} = -\Big[ \log\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) + \sum_{k=1}^K \log\big(1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})\big) \Big]
$$

## 2. 负样本采样方式（噪声分布）
1. 噪声分布 $P_n(w)$：通常采用词频的3/4次幂归一化
    $$
    P_n(w) = \dfrac{\text{count}(w)^{3/4}}{\sum_{w'\in V}\text{count}(w')^{3/4}}
    $$
2. 采样流程：
    - 统计全部词汇的词频，构建噪声分布；
    - 每次训练样本时，从 $P_n(w)$ 中独立抽取 $K$ 个不同于 $w_o,w_c$ 的词作为负样本；
    - 高频词更容易被选为负样本，降低计算开销，替代完整Softmax。

In [4]:
import numpy as np

# ===================== 5.2 CBOW模型前向+完整Softmax交叉熵损失 =====================
print("===== CBOW 前向传播 & 完整Softmax损失计算 =====")

def cbow_forward_loss(context_indices_batch, target_indices_batch, W, W_out):
    """
    CBOW 前向传播 + 完整Softmax交叉熵损失（无负采样）
    参数：
        context_indices_batch: 批次上下文索引，shape (batch_size, context_size)
        target_indices_batch: 批次中心词索引，shape (batch_size,)
        W: 输入嵌入矩阵 (V, d)，V词汇总数，d嵌入维度
        W_out: 输出权重矩阵 (d, V)
    返回：
        avg_loss: 批次平均交叉熵损失标量
    """
    batch_size, context_size = context_indices_batch.shape
    V, d = W.shape

    # 步骤1：取出所有上下文词向量，按样本求平均得到隐藏层
    # context_embeds: (batch, context_size, d)
    context_embeds = W[context_indices_batch]
    h = np.mean(context_embeds, axis=1)  # h: (batch, d)

    # 步骤2：计算得分 logits = h @ W_out，shape (batch, V)
    logits = h @ W_out

    # 步骤3：完整Softmax归一化，防止指数溢出
    max_vals = np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits - max_vals)
    softmax_prob = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # 步骤4：取出目标词对应概率，计算交叉熵损失 -log(p_target)
    batch_idx = np.arange(batch_size)
    p_target = softmax_prob[batch_idx, target_indices_batch]
    ce_loss = -np.log(p_target + 1e-10)  # 加极小值防log(0)

    # 批次平均损失
    avg_loss = np.mean(ce_loss)
    return avg_loss

# ---------------------- 测试用例 ----------------------
if __name__ == "__main__":
    # 超参
    V = 10    # 词汇表大小
    d = 4     # 嵌入维度
    batch_size = 3
    context_size = 2

    # 随机初始化权重
    W = np.random.randn(V, d) * 0.1
    W_out = np.random.randn(d, V) * 0.1

    # 构造输入：每个样本2个上下文词，对应1个中心词
    context_batch = np.array([[0, 2], [1, 3], [5, 7]])
    target_batch = np.array([1, 2, 6])

    loss = cbow_forward_loss(context_batch, target_batch, W, W_out)
    print(f"批次平均CBOW交叉熵损失: {loss:.4f}")
    print("\n✅ CBOW前向传播与损失计算完成！")

===== CBOW 前向传播 & 完整Softmax损失计算 =====
批次平均CBOW交叉熵损失: 2.3090

✅ CBOW前向传播与损失计算完成！


# 6.1 缩放点积注意力分步计算
## 已知维度
Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)
dk = 4，√dk = 2
缩放得分公式：score = QK^T / √dk
无掩码缩放点积注意力完整三步：
1. 计算注意力得分矩阵 score = QK^T / √dk
2. 对得分矩阵每行做softmax，得到注意力权重 attn_weights
3. 输出 Output = attn_weights · V

## 步骤1：计算得分矩阵
K^T ∈ R^(4×3)
矩阵乘积 QK^T ∈ R^(2×3)
score = QK^T / 2 ，维度 R^(2×3)
- 2行对应查询数量（2个query），3列对应键/值数量（3组key-value对）

## 步骤2：Softmax归一化注意力权重
对得分矩阵每一行独立执行softmax：
attn_{i,j} = exp(score_{i,j}) / [ exp(score_{i,1}) + exp(score_{i,2}) + exp(score_{i,3}) ]
attn_weights 维度 R^(2×3)，每行所有权重相加等于1。

## 步骤3：加权求和得到输出
Output = attn_weights · V
attn_weights(2×3) 乘 V(3×5)，矩阵相乘维度匹配。

## 最终输出尺寸结论
缩放点积注意力输出矩阵形状为 2 × 5。

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ===================== 6.2 多头注意力前向传播实现 =====================
print("===== 多头注意力 Multi-Head Attention 前向传播 =====")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 单头维度 d_k=2
        
        # Q/K/V 投影层
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        # 输出融合线性层
        self.w_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attn(self, q, k, v):
        dk = q.size(-1)
        score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(dk, dtype=torch.float32))
        attn_weight = F.softmax(score, dim=-1)
        out = torch.matmul(attn_weight, v)
        return out

    def split_heads(self, x):
        # x: (seq_len, batch, d_model)
        seq_len, batch, _ = x.shape
        return x.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)

    def concat_heads(self, x):
        # x: (num_heads, seq_len, batch, d_k)
        heads, seq_len, batch, _ = x.shape
        return x.permute(1, 2, 0, 3).reshape(seq_len, batch, self.d_model)

    def forward(self, X):
        """
        参数：
            X: 输入序列 (seq_len, batch, d_model)
        返回：
            out: 多头注意力输出，shape与输入完全一致 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        # 1. 线性投影 Q,K,V
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)

        # 2. 分头
        Q = self.split_heads(Q)  # (heads, seq_len, batch, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # 3. 每头独立缩放点积注意力
        attn_out = self.scaled_dot_product_attn(Q, K, V)

        # 4. 拼接所有头
        concat_out = self.concat_heads(attn_out)

        # 5. 最终线性层融合
        out = self.w_o(concat_out)
        return out

# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    seq_len = 6
    batch = 3
    d_model = 4
    num_heads = 2

    X = torch.randn(seq_len, batch, d_model)
    mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
    output = mha(X)

    print(f"输入X shape: {X.shape}")
    print(f"多头注意力输出 shape: {output.shape}")
    print("\n✅ 多头注意力前向传播完成，输出维度与输入完全匹配！")

===== 多头注意力 Multi-Head Attention 前向传播 =====
输入X shape: torch.Size([6, 3, 4])
多头注意力输出 shape: torch.Size([6, 3, 4])

✅ 多头注意力前向传播完成，输出维度与输入完全匹配！
